# TerraMind Base Flood Fine-tuning จาก Sen1Floods11

Notebook นี้ใช้สำหรับสร้าง checkpoint ของโมเดล **TerraMind Base** สำหรับงาน flood / water segmentation  
โดย fine-tune จากชุดข้อมูลสาธารณะ **Sen1Floods11 v1.1** แล้วบันทึก checkpoint ไปยัง Google Drive

checkpoint ที่ได้จาก notebook นี้สามารถนำไปใช้ต่อกับ prediction notebook ได้ทันที

## สิ่งที่ notebook นี้ทำ

1. ตรวจสอบ GPU และติดตั้ง library ที่จำเป็น
2. ดาวน์โหลดชุดข้อมูล Sen1Floods11
3. ตรวจสอบโครงสร้างข้อมูล S1, S2 และ label
4. สร้าง config สำหรับ TerraTorch
5. Fine-tune `terramind_v1_base`
6. เลือก checkpoint ที่ดีที่สุดจาก `val/IoU_Flood`
7. copy checkpoint ไปยัง Google Drive
8. ทดสอบ checkpoint กับ test split
9. แสดงตัวอย่าง prediction เพื่อยืนยันว่าโมเดลใช้งานได้

> คำแนะนำ: ใช้ Google Colab GPU เช่น T4, L4 หรือ A100  
> ถ้าใช้ T4 ค่าเริ่มต้นใน notebook นี้ถูกตั้งให้เหมาะกับหน่วยความจำประมาณ 15 GB แล้ว

# 0 — โฟลเดอร์ผลลัพธ์ที่ notebook จะสร้าง

ค่าเริ่มต้นจะบันทึก checkpoint ไว้ใน Google Drive ที่:

```text
MyDrive/GeoFM_Projects/
└── Flood_Prediction_Package/
    ├── checkpoints/
    │   └── best_terramind_base_sen1floods11.ckpt
    ├── configs/
    │   └── terramind_base_sen1floods11.yaml
    ├── logs/
    │   ├── terramind_training.log
    │   └── training_summary.json
    └── previews/
        └── ตัวอย่างภาพ prediction จาก test split
```

ไฟล์สำคัญที่สุดคือ:

```text
best_terramind_base_sen1floods11.ckpt
```

ให้นำไฟล์นี้ไปใช้ใน prediction notebook

In [ ]:
# ============================================================
# 1 — ติดตั้ง TerraTorch บน Google Colab
# ============================================================
# ก่อนรัน:
# Runtime → Change runtime type → GPU
#
# วิธีนี้จะไม่ให้ pip เปลี่ยน PyTorch, torchvision และ NumPy
# ที่ Colab ติดตั้งมาให้
# ============================================================

import os
import sys
import subprocess
import torch


print("=" * 80)
print("COLAB ENVIRONMENT")
print("=" * 80)
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())


if not torch.cuda.is_available():
    raise RuntimeError(
        "ไม่พบ GPU กรุณาเลือก "
        "Runtime → Change runtime type → GPU"
    )


def run_pip(packages, no_deps=False):
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
    ]

    if no_deps:
        command.append("--no-deps")

    command.extend(packages)

    print("\n" + "-" * 80)
    print("กำลังติดตั้ง:")
    print(" ".join(command))
    print("-" * 80)

    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            "ติดตั้ง package ไม่สำเร็จ\n"
            "รายละเอียดจาก pip แสดงอยู่ด้านบน"
        )


# ------------------------------------------------------------
# 1. อัปเดตเครื่องมือติดตั้ง
# ------------------------------------------------------------

run_pip([
    "--upgrade",
    "pip",
    "wheel",
    "setuptools<81",
])


# ------------------------------------------------------------
# 2. ติดตั้ง TerraTorch โดยไม่ให้ resolver เปลี่ยน PyTorch
# ------------------------------------------------------------

run_pip(
    ["terratorch==1.2.8"],
    no_deps=True,
)


# ------------------------------------------------------------
# 3. ติดตั้ง dependency ที่ใช้จริง
#
# ไม่ระบุ torch, torchvision และ numpy
# เพื่อใช้ชุด GPU ที่ Colab เตรียมไว้
# ------------------------------------------------------------

run_pip([
    "lightning<2.6",
    "torchmetrics",
    "torchgeo",
    "timm",
    "segmentation-models-pytorch",
    "kornia",
    "einops",
    "jsonargparse[signatures]",
    "albumentations",
    "huggingface-hub",
    "tensorboard",
    "pyyaml",
    "rich",
])


# ------------------------------------------------------------
# 4. ตรวจว่า pip มองเห็น TerraTorch
#
# ยังไม่ import ใน session นี้ เพราะ package เพิ่งถูกติดตั้ง
# ------------------------------------------------------------

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "show",
        "terratorch",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("\n" + "=" * 80)
print("INSTALLATION RESULT")
print("=" * 80)
print(result.stdout)

if result.returncode != 0:
    raise RuntimeError("pip ยังไม่พบ TerraTorch หลังติดตั้ง")


print("ติดตั้งเสร็จแล้ว")
print("กำลัง restart runtime อัตโนมัติ")
print("หลังเชื่อมต่อใหม่ ให้เริ่มรันจาก Cell 2")

os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# ติดตั้ง dependency ที่ TerraTorch ยังขาด
# ============================================================

import sys
import subprocess

packages = [
    "rioxarray",
    "xarray",
    "dask",
    "netcdf4",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    *packages,
]

print("กำลังติดตั้ง:")
print(" ".join(packages))

subprocess.check_call(command)

print("\nติดตั้งเสร็จแล้ว")
print("กรุณา Restart session แล้วรันต่อจาก Cell 2")

In [ ]:
# ============================================================
# 3 — Mount Google Drive และตั้งค่าโฟลเดอร์
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")

# ปรับ path นี้ได้ถ้าโฟลเดอร์โครงการของผู้ใช้ชื่อไม่เหมือนกัน
PROJECT_ROOT = DRIVE_ROOT / "GeoFM_Projects"

PACKAGE_ROOT = PROJECT_ROOT / "Flood_Prediction_Package"

CHECKPOINT_DIR = PACKAGE_ROOT / "checkpoints"
CONFIG_DIR = PACKAGE_ROOT / "configs"
LOG_DIR = PACKAGE_ROOT / "logs"
PREVIEW_DIR = PACKAGE_ROOT / "previews"

# ใช้ local disk ของ Colab สำหรับ training จะเร็วกว่า Drive
WORK_ROOT = Path("/content/TerraMind_Flood_Training")
DATA_ROOT = WORK_ROOT / "sen1floods11_v1.1"
OUTPUT_ROOT = WORK_ROOT / "outputs"
LOCAL_CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOCAL_CONFIG_DIR = WORK_ROOT / "configs"

for folder in [
    PACKAGE_ROOT,
    CHECKPOINT_DIR,
    CONFIG_DIR,
    LOG_DIR,
    PREVIEW_DIR,
    WORK_ROOT,
    OUTPUT_ROOT,
    LOCAL_CHECKPOINT_DIR,
    LOCAL_CONFIG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = LOCAL_CONFIG_DIR / "terramind_base_sen1floods11.yaml"
TRAIN_LOG_PATH = LOG_DIR / "terramind_training.log"
DRIVE_CONFIG_PATH = CONFIG_DIR / CONFIG_PATH.name
DRIVE_BEST_CKPT = CHECKPOINT_DIR / "best_terramind_base_sen1floods11.ckpt"
DRIVE_LAST_CKPT = CHECKPOINT_DIR / "last_terramind_base_sen1floods11.ckpt"
SUMMARY_JSON = LOG_DIR / "training_summary.json"

print("=" * 100)
print("PROJECT PATHS")
print("=" * 100)
print("PROJECT_ROOT        :", PROJECT_ROOT)
print("PACKAGE_ROOT        :", PACKAGE_ROOT)
print("WORK_ROOT           :", WORK_ROOT)
print("DATA_ROOT           :", DATA_ROOT)
print("OUTPUT_ROOT         :", OUTPUT_ROOT)
print("CONFIG_PATH         :", CONFIG_PATH)
print("DRIVE_BEST_CKPT     :", DRIVE_BEST_CKPT)

In [ ]:
# ============================================================
# 4 — ตั้งค่า training
# ============================================================

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# เลือกโหมดการ train
# - quick: รันเร็ว ใช้ตรวจ workflow และได้ checkpoint เบื้องต้น
# - good : ค่าแนะนำสำหรับใช้งานจริงบน T4
# - full : รันนานขึ้น เหมาะกับ GPU ที่แรงกว่า
RUN_MODE = "good"

if RUN_MODE == "quick":
    MAX_EPOCHS = 5
    LIMIT_TRAIN_BATCHES = 0.35
    LIMIT_VAL_BATCHES = 1.0
elif RUN_MODE == "good":
    MAX_EPOCHS = 20
    LIMIT_TRAIN_BATCHES = 1.0
    LIMIT_VAL_BATCHES = 1.0
elif RUN_MODE == "full":
    MAX_EPOCHS = 35
    LIMIT_TRAIN_BATCHES = 1.0
    LIMIT_VAL_BATCHES = 1.0
else:
    raise ValueError("RUN_MODE ต้องเป็น quick, good หรือ full")

# ค่านี้เคยรันผ่านบน Tesla T4 15GB
BATCH_SIZE = 2
ACCUMULATE_GRAD_BATCHES = 4

# ถ้าเจอ CUDA out of memory ให้ใช้:
# BATCH_SIZE = 1
# ACCUMULATE_GRAD_BATCHES = 8 หรือ 12

NUM_WORKERS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PRECISION = "16-mixed"

# ตรวจ GPU อีกครั้งใน cell นี้
if not torch.cuda.is_available():
    raise RuntimeError(
        "ไม่พบ GPU กรุณาเลือก Runtime → Change runtime type → GPU"
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("=" * 100)
print("TRAINING SETTINGS")
print("=" * 100)
print("RUN_MODE                :", RUN_MODE)
print("MAX_EPOCHS              :", MAX_EPOCHS)
print("BATCH_SIZE              :", BATCH_SIZE)
print("ACCUMULATE_GRAD_BATCHES :", ACCUMULATE_GRAD_BATCHES)
print("Effective batch size    :", BATCH_SIZE * ACCUMULATE_GRAD_BATCHES)
print("NUM_WORKERS             :", NUM_WORKERS)
print("LEARNING_RATE           :", LEARNING_RATE)
print("PRECISION               :", PRECISION)
print("GPU                     :", GPU_NAME)
print("GPU memory              :", f"{GPU_MEMORY_GB:.2f} GB")

print("=" * 100)
print("TRAINING SETTINGS")
print("=" * 100)
print("RUN_MODE                :", RUN_MODE)
print("MAX_EPOCHS              :", MAX_EPOCHS)
print("BATCH_SIZE              :", BATCH_SIZE)
print("ACCUMULATE_GRAD_BATCHES :", ACCUMULATE_GRAD_BATCHES)
print("Effective batch size    :", BATCH_SIZE * ACCUMULATE_GRAD_BATCHES)
print("NUM_WORKERS             :", NUM_WORKERS)
print("LEARNING_RATE           :", LEARNING_RATE)
print("PRECISION               :", PRECISION)
print("GPU                     :", GPU_NAME)

In [ ]:
# ============================================================
# 5 — ดาวน์โหลด Sen1Floods11 v1.1
# ============================================================

from pathlib import Path
from huggingface_hub import hf_hub_download
import tarfile
import os

def find_sen1floods_root(search_root):
    for root, dirs, files in os.walk(search_root):
        root = Path(root)
        if (
            (root / "data" / "S2L1CHand").exists()
            and (root / "data" / "S1GRDHand").exists()
            and (root / "data" / "LabelHand").exists()
            and (root / "splits" / "flood_train_data.txt").exists()
        ):
            return root
    return None

existing_root = find_sen1floods_root(WORK_ROOT)

if existing_root is not None:
    DATA_ROOT = existing_root
    print("พบ dataset เดิม:", DATA_ROOT)

else:
    print("กำลังดาวน์โหลด Sen1Floods11 v1.1 ...")

    archive_path = hf_hub_download(
        repo_id="blumenstiel/Sen1Floods11",
        filename="sen1floods11_v1_1.tar.gz",
        repo_type="dataset",
        local_dir=str(WORK_ROOT),
    )

    archive_path = Path(archive_path)
    print("Downloaded:", archive_path)

    print("กำลังแตกไฟล์ ...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(WORK_ROOT)

    DATA_ROOT = find_sen1floods_root(WORK_ROOT)

    if DATA_ROOT is None:
        raise FileNotFoundError("แตกไฟล์แล้วแต่ไม่พบโครงสร้าง Sen1Floods11")

print("=" * 100)
print("DATASET READY")
print("=" * 100)
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
# ============================================================
# 6 — ตรวจสอบข้อมูลและ split
# ============================================================

import pandas as pd
import rasterio
import numpy as np
from IPython.display import display

S2_DIR = DATA_ROOT / "data" / "S2L1CHand"
S1_DIR = DATA_ROOT / "data" / "S1GRDHand"
LABEL_DIR = DATA_ROOT / "data" / "LabelHand"

TRAIN_SPLIT = DATA_ROOT / "splits" / "flood_train_data.txt"
VAL_SPLIT = DATA_ROOT / "splits" / "flood_valid_data.txt"
TEST_SPLIT = DATA_ROOT / "splits" / "flood_test_data.txt"

def read_split(path):
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]

train_ids = read_split(TRAIN_SPLIT)
val_ids = read_split(VAL_SPLIT)
test_ids = read_split(TEST_SPLIT)

print("Train samples:", len(train_ids))
print("Val samples  :", len(val_ids))
print("Test samples :", len(test_ids))

# ตรวจจำนวนไฟล์
s2_files = sorted(S2_DIR.glob("*_S2Hand.tif"))
s1_files = sorted(S1_DIR.glob("*_S1Hand.tif"))
label_files = sorted(LABEL_DIR.glob("*_LabelHand.tif"))

print("S2 files    :", len(s2_files))
print("S1 files    :", len(s1_files))
print("Label files :", len(label_files))

# ตรวจ metadata ตัวอย่าง
sample_id = train_ids[0]

sample_paths = {
    "S2L1C": S2_DIR / f"{sample_id}_S2Hand.tif",
    "S1GRD": S1_DIR / f"{sample_id}_S1Hand.tif",
    "Label": LABEL_DIR / f"{sample_id}_LabelHand.tif",
}

rows = []

for name, path in sample_paths.items():
    with rasterio.open(path) as src:
        arr = src.read()
        rows.append(
            {
                "name": name,
                "bands": src.count,
                "shape": f"{src.height} x {src.width}",
                "dtype": src.dtypes[0],
                "crs": str(src.crs),
                "min": float(np.nanmin(arr)),
                "max": float(np.nanmax(arr)),
            }
        )

display(pd.DataFrame(rows))

In [ ]:
# ============================================================
# 7 — Preview ตัวอย่างข้อมูล
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt

def stretch_rgb(rgb):
    rgb = rgb.astype(np.float32)
    lo = np.percentile(rgb, 2, axis=(0, 1), keepdims=True)
    hi = np.percentile(rgb, 98, axis=(0, 1), keepdims=True)
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

def show_sample(sample_id):
    with rasterio.open(S2_DIR / f"{sample_id}_S2Hand.tif") as src:
        s2 = src.read().astype(np.float32)

    with rasterio.open(S1_DIR / f"{sample_id}_S1Hand.tif") as src:
        s1 = src.read().astype(np.float32)

    with rasterio.open(LABEL_DIR / f"{sample_id}_LabelHand.tif") as src:
        label = src.read(1)

    rgb = stretch_rgb(np.moveaxis(s2[[3, 2, 1]], 0, -1))

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(rgb)
    axes[0].set_title("Sentinel-2 RGB")

    axes[1].imshow(s1[0], cmap="gray", vmin=-25, vmax=0)
    axes[1].set_title("Sentinel-1 VV")

    axes[2].imshow(s1[1], cmap="gray", vmin=-30, vmax=-5)
    axes[2].set_title("Sentinel-1 VH")

    axes[3].imshow(label, cmap="Blues", vmin=0, vmax=1)
    axes[3].set_title("Label")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(sample_id)
    plt.tight_layout()
    plt.show()

for sid in train_ids[:3]:
    show_sample(sid)

In [ ]:
# ============================================================
# 8 — สร้าง TerraTorch config
# ============================================================

import yaml
from pathlib import Path

S2_MEAN = [
    2357.089, 2137.385, 2018.788, 2082.986, 2295.651,
    2854.537, 3122.849, 3040.560, 3306.481, 1473.847,
    506.070, 2472.825, 1838.929,
]

S2_STD = [
    1624.683, 1675.806, 1557.708, 1833.702, 1823.738,
    1733.977, 1732.131, 1679.732, 1727.260, 1024.687,
    442.165, 1331.411, 1160.419,
]

S1_MEAN = [-12.599, -20.293]
S1_STD = [5.195, 5.890]

MODEL_ARGS = {
    "backbone": "terramind_v1_base",
    "backbone_pretrained": True,
    "backbone_modalities": ["S2L1C", "S1GRD"],
    "backbone_merge_method": "mean",
    "necks": [
        {"name": "SelectIndices", "indices": [2, 5, 8, 11]},
        {"name": "ReshapeTokensToImage", "remove_cls_token": False},
        {"name": "LearnedInterpolateToPyramidal"},
    ],
    "decoder": "UNetDecoder",
    "decoder_channels": [512, 256, 128, 64],
    "head_dropout": 0.1,
    "num_classes": 2,
}

callbacks = [
    {
        "class_path": "lightning.pytorch.callbacks.ModelCheckpoint",
        "init_args": {
            "dirpath": str(LOCAL_CHECKPOINT_DIR),
            "filename": "best_terramind_base_sen1floods11",
            "monitor": "val/IoU_Flood",
            "mode": "max",
            "save_top_k": 1,
            "save_last": True,
            "save_weights_only": False,
            "auto_insert_metric_name": False,
        },
    },
    {
        "class_path": "lightning.pytorch.callbacks.EarlyStopping",
        "init_args": {
            "monitor": "val/IoU_Flood",
            "mode": "max",
            "patience": 6,
            "min_delta": 0.001,
            "check_on_train_epoch_end": False,
            "verbose": True,
        },
    },
    {
        "class_path": "lightning.pytorch.callbacks.LearningRateMonitor",
        "init_args": {"logging_interval": "epoch"},
    },
]

config = {
    "seed_everything": SEED,

    "trainer": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": PRECISION,
        "max_epochs": MAX_EPOCHS,
        "accumulate_grad_batches": ACCUMULATE_GRAD_BATCHES,
        "gradient_clip_val": 1.0,
        "limit_train_batches": LIMIT_TRAIN_BATCHES,
        "limit_val_batches": LIMIT_VAL_BATCHES,
        "limit_test_batches": 1.0,
        "num_sanity_val_steps": 2,
        "log_every_n_steps": 5,
        "enable_checkpointing": True,
        "default_root_dir": str(OUTPUT_ROOT),
        "callbacks": callbacks,
    },

    "data": {
        "class_path": "terratorch.datamodules.GenericMultiModalDataModule",
        "init_args": {
            "task": "segmentation",
            "batch_size": BATCH_SIZE,
            "num_workers": NUM_WORKERS,
            "modalities": ["S2L1C", "S1GRD"],
            "rgb_modality": "S2L1C",
            "rgb_indices": [3, 2, 1],

            "train_data_root": {"S2L1C": str(S2_DIR), "S1GRD": str(S1_DIR)},
            "train_label_data_root": str(LABEL_DIR),

            "val_data_root": {"S2L1C": str(S2_DIR), "S1GRD": str(S1_DIR)},
            "val_label_data_root": str(LABEL_DIR),

            "test_data_root": {"S2L1C": str(S2_DIR), "S1GRD": str(S1_DIR)},
            "test_label_data_root": str(LABEL_DIR),

            "train_split": str(TRAIN_SPLIT),
            "val_split": str(VAL_SPLIT),
            "test_split": str(TEST_SPLIT),

            "image_grep": {"S2L1C": "*_S2Hand.tif", "S1GRD": "*_S1Hand.tif"},
            "label_grep": "*_LabelHand.tif",

            "no_label_replace": -1,
            "no_data_replace": 0,
            "num_classes": 2,

            "means": {"S2L1C": S2_MEAN, "S1GRD": S1_MEAN},
            "stds": {"S2L1C": S2_STD, "S1GRD": S1_STD},

            "train_transform": [
                {"class_path": "albumentations.D4", "init_args": {"p": 1.0}},
                {"class_path": "albumentations.pytorch.transforms.ToTensorV2", "init_args": {}},
            ],
            "val_transform": [
                {"class_path": "albumentations.pytorch.transforms.ToTensorV2", "init_args": {}},
            ],
            "test_transform": [
                {"class_path": "albumentations.pytorch.transforms.ToTensorV2", "init_args": {}},
            ],
        },
    },

    "model": {
        "class_path": "terratorch.tasks.SemanticSegmentationTask",
        "init_args": {
            "model_factory": "EncoderDecoderFactory",
            "model_args": MODEL_ARGS,
            "loss": "dice",
            "ignore_index": -1,
            "freeze_backbone": False,
            "freeze_decoder": False,
            "class_names": ["Others", "Flood"],
        },
    },

    "optimizer": {
        "class_path": "torch.optim.AdamW",
        "init_args": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    },

    "lr_scheduler": {
        "class_path": "torch.optim.lr_scheduler.CosineAnnealingLR",
        "init_args": {"T_max": MAX_EPOCHS, "eta_min": 1e-7},
    },
}

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print("=" * 100)
print("CONFIG CREATED")
print("=" * 100)
print(CONFIG_PATH)
print("\nMonitor metric: val/IoU_Flood")
print("Checkpoint dir:", LOCAL_CHECKPOINT_DIR)

In [ ]:
# ============================================================
# 9 — Fine-tune TerraMind Base
# ============================================================
# แสดง log แบบ real-time ใน Colab
# ============================================================

import os
import time
import subprocess

start_time = time.time()

command = [
    "terratorch",
    "fit",
    "-c",
    str(CONFIG_PATH),
]

print("=" * 100)
print("START TRAINING")
print("=" * 100)
print(" ".join(command))
print("=" * 100)

env = os.environ.copy()

# บังคับ Python และ Lightning ส่ง output ทันที
env["PYTHONUNBUFFERED"] = "1"
env["TERM"] = "xterm-256color"

with TRAIN_LOG_PATH.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in iter(process.stdout.readline, ""):

        print(
            line,
            end="",
            flush=True,
        )

        log_file.write(line)
        log_file.flush()

    process.stdout.close()

    return_code = process.wait()


elapsed_minutes = (
    time.time() - start_time
) / 60


if return_code != 0:
    raise RuntimeError(
        "Training ไม่สำเร็จ\n"
        f"ตรวจ log ได้ที่:\n{TRAIN_LOG_PATH}"
    )


print("\n" + "=" * 100)
print("TRAINING COMPLETED")
print("=" * 100)

print(
    "Elapsed time:",
    f"{elapsed_minutes:.2f} minutes",
)

print(
    "Training log:",
    TRAIN_LOG_PATH,
)

In [ ]:
# ============================================================
# 10 — เลือก checkpoint ที่ดีที่สุดและ copy ไป Drive
# ============================================================

import shutil
import json
import time
from pathlib import Path

checkpoints = sorted(
    LOCAL_CHECKPOINT_DIR.rglob("*.ckpt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not checkpoints:
    raise FileNotFoundError(f"ไม่พบ checkpoint ใน {LOCAL_CHECKPOINT_DIR}")

print("=" * 100)
print("AVAILABLE CHECKPOINTS")
print("=" * 100)

for p in checkpoints:
    print(f"{p.stat().st_size / 1024**2:8.2f} MB | {p}")

best_candidates = [
    p for p in checkpoints
    if "last" not in p.name.lower()
]

BEST_CKPT = best_candidates[0] if best_candidates else checkpoints[0]
LAST_CKPT = next((p for p in checkpoints if p.name == "last.ckpt"), None)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_CKPT, DRIVE_BEST_CKPT)
shutil.copy2(CONFIG_PATH, DRIVE_CONFIG_PATH)

if LAST_CKPT is not None:
    shutil.copy2(LAST_CKPT, DRIVE_LAST_CKPT)

training_summary = {
    "model": "terramind_v1_base",
    "dataset": "Sen1Floods11 v1.1",
    "modalities": ["S2L1C", "S1GRD"],
    "classes": ["Others", "Flood"],
    "monitor_metric": "val/IoU_Flood",
    "run_mode": RUN_MODE,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "accumulate_grad_batches": ACCUMULATE_GRAD_BATCHES,
    "learning_rate": LEARNING_RATE,
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "best_checkpoint_local": str(BEST_CKPT),
    "best_checkpoint_drive": str(DRIVE_BEST_CKPT),
    "config_drive": str(DRIVE_CONFIG_PATH),
    "training_log": str(TRAIN_LOG_PATH),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_args": MODEL_ARGS,
}

SUMMARY_JSON.write_text(
    json.dumps(training_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("=" * 100)
print("CHECKPOINT SAVED")
print("=" * 100)
print("Best checkpoint:", DRIVE_BEST_CKPT)
print("Config         :", DRIVE_CONFIG_PATH)
print("Summary        :", SUMMARY_JSON)
print("Size           :", f"{DRIVE_BEST_CKPT.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# ============================================================
# 11 — ทดสอบ checkpoint กับ test split
# ============================================================

import subprocess

command = (
    f'terratorch test -c "{CONFIG_PATH}" --ckpt_path "{BEST_CKPT}"'
)

print("=" * 100)
print("TEST CHECKPOINT")
print("=" * 100)
print(command)
print("=" * 100)

result = subprocess.run(command, shell=True, executable="/bin/bash")

if result.returncode != 0:
    raise RuntimeError("Test failed กรุณาดูข้อความ error ด้านบน")

print("Test completed")

In [ ]:
# ============================================================
# 12 — แสดงตัวอย่าง prediction จาก checkpoint
# ============================================================

from terratorch.tasks import SemanticSegmentationTask
import torch
import numpy as np
import rasterio
import matplotlib.pyplot as plt

preview_model_args = MODEL_ARGS.copy()
preview_model_args["backbone_pretrained"] = False

model = SemanticSegmentationTask.load_from_checkpoint(
    checkpoint_path=str(BEST_CKPT),
    map_location=DEVICE,
    model_factory="EncoderDecoderFactory",
    model_args=preview_model_args,
)

model = model.to(DEVICE)
model.eval()

S2_MEAN_ARR = np.array(S2_MEAN, dtype=np.float32)
S2_STD_ARR = np.array(S2_STD, dtype=np.float32)
S1_MEAN_ARR = np.array(S1_MEAN, dtype=np.float32)
S1_STD_ARR = np.array(S1_STD, dtype=np.float32)

def infer_sample(sample_id):
    with rasterio.open(S2_DIR / f"{sample_id}_S2Hand.tif") as src:
        s2 = src.read().astype(np.float32)

    with rasterio.open(S1_DIR / f"{sample_id}_S1Hand.tif") as src:
        s1 = src.read().astype(np.float32)

    with rasterio.open(LABEL_DIR / f"{sample_id}_LabelHand.tif") as src:
        label = src.read(1)

    s2_norm = (s2 - S2_MEAN_ARR[:, None, None]) / S2_STD_ARR[:, None, None]
    s1_norm = (s1 - S1_MEAN_ARR[:, None, None]) / S1_STD_ARR[:, None, None]

    batch = {
        "S2L1C": torch.from_numpy(s2_norm[None]).to(DEVICE),
        "S1GRD": torch.from_numpy(s1_norm[None]).to(DEVICE),
    }

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        result = model(batch)
        logits = result.output if hasattr(result, "output") else result
        prob = torch.softmax(logits, dim=1)[0, 1].float().cpu().numpy()

    return s2, s1, label, prob

preview_ids = test_ids[:4]

for sample_id in preview_ids:
    s2, s1, label, prob = infer_sample(sample_id)
    pred = (prob >= 0.5).astype(np.uint8)

    rgb = stretch_rgb(np.moveaxis(s2[[3, 2, 1]], 0, -1))

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    axes[0].imshow(rgb)
    axes[0].set_title("S2 RGB")

    axes[1].imshow(s1[0], cmap="gray", vmin=-25, vmax=0)
    axes[1].set_title("S1 VV")

    axes[2].imshow(label, cmap="Blues", vmin=0, vmax=1)
    axes[2].set_title("Label")

    im = axes[3].imshow(prob, cmap="viridis", vmin=0, vmax=1)
    axes[3].set_title("Flood probability")

    axes[4].imshow(rgb)
    axes[4].imshow(np.ma.masked_where(pred == 0, pred), cmap="autumn", alpha=0.55)
    axes[4].set_title("Prediction overlay")

    for ax in axes:
        ax.axis("off")

    plt.colorbar(im, ax=axes[3], fraction=0.046)
    plt.suptitle(sample_id)
    plt.tight_layout()

    out_png = PREVIEW_DIR / f"{sample_id}_prediction_preview.png"
    plt.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.show()

print("Previews saved to:", PREVIEW_DIR)

In [ ]:
# ============================================================
# 13 — ตรวจ output สุดท้าย
# ============================================================

print("=" * 100)
print("FINAL OUTPUT CHECK")
print("=" * 100)

final_files = [
    DRIVE_BEST_CKPT,
    DRIVE_CONFIG_PATH,
    SUMMARY_JSON,
    TRAIN_LOG_PATH,
]

if DRIVE_LAST_CKPT.exists():
    final_files.append(DRIVE_LAST_CKPT)

for path in final_files:
    print(path)
    print("  exists:", path.exists())
    if path.exists():
        print("  size  :", f"{path.stat().st_size / 1024**2:.2f} MB")

print("\nนำไฟล์นี้ไปใช้ใน prediction notebook:")
print(DRIVE_BEST_CKPT)

# 14 — คำอธิบายสำหรับผู้ที่จะนำไปใช้ต่อ

## checkpoint ที่ต้องใช้

หลังจากรันจบ ให้ใช้ไฟล์นี้:

```text
Flood_Prediction_Package/checkpoints/best_terramind_base_sen1floods11.ckpt
```

## ทำไมใช้ `val/IoU_Flood`

งาน flood detection สนใจ class น้ำท่วมโดยตรง จึงเลือก checkpoint จาก metric:

```text
val/IoU_Flood
```

ไม่ใช้ overall accuracy เพราะน้ำท่วมมักเป็น class ที่มีพื้นที่น้อยกว่า background มาก

## ข้อจำกัด

checkpoint นี้ fine-tune จาก Sen1Floods11 ซึ่งเป็น dataset สาธารณะ  
เมื่อนำไปใช้กับพื้นที่อื่น เช่น พื้นที่ศึกษาของคุณ ควรตีความผลลัพธ์เป็น prototype หรือ probable flood extent ก่อน  
ถ้ามี label ของพื้นที่จริงในอนาคต ควรนำมา fine-tune ต่อเพื่อเพิ่มความแม่นยำเฉพาะพื้นที่

## การนำไปใช้กับ prediction notebook

prediction notebook คาดหวัง checkpoint ชื่อนี้:

```text
best_terramind_base_sen1floods11.ckpt
```

และควรอยู่ใน:

```text
Flood_Prediction_Package/checkpoints/
```